# Prevedere l'abbandono dei clienti (Telco Customer Churn)

## 0. Il problema

Un'azienda di telecomunicazioni perde ogni mese una parte dei suoi clienti: disdicono il contratto e passano alla concorrenza. Questo fenomeno si chiama **churn** (abbandono).

Trovare un nuovo cliente costa molto più che tenerne uno esistente. Se sapessimo **in anticipo** chi sta per andarsene, potremmo fare un'offerta mirata, chiamarlo, proporgli un contratto migliore.

**Obiettivo:** dato un cliente (tipo di contratto, servizi attivi, quanto paga, da quanto tempo è cliente…), prevedere la **probabilità che abbandoni**.

In termini di machine learning:
- È una **classificazione binaria**: la risposta è *Sì, abbandona* (1) oppure *No* (0).

## Il dataset

Usiamo il dataset **Telco Customer Churn** pubblicato da IBM (molto noto anche su Kaggle). Ogni **riga è un cliente**, ogni **colonna una sua caratteristica**: **7043 clienti** e **21 colonne**.

| Gruppo | Colonne | Esempio di valori |
|---|---|---|
| Identificativo | `customerID` | `7590-VHVEG` (non serve al modello) |
| Dati anagrafici | `gender`, `SeniorCitizen`, `Partner`, `Dependents` | `Female`, `0`, `Yes`, `No` |
| Rapporto con l'azienda | `tenure` (mesi da cliente), `Contract`, `PaperlessBilling`, `PaymentMethod` | `1`, `Month-to-month`, `Yes`, `Electronic check` |
| Servizi telefonici | `PhoneService`, `MultipleLines` | `Yes`, `No phone service` |
| Servizi internet | `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | `Fiber optic`, `No internet service` |
| Costi | `MonthlyCharges`, `TotalCharges` | `29.85`, `1889.5` |
| **Target** (da prevedere) | `Churn` | `Yes` / `No` |

## Il piano della lezione

1. Importare le librerie
2. Caricare ed esplorare i dati
3. Preparare i dati: pulizia, codifica, divisione train/validation/test, normalizzazione
4. **Modello da zero con NumPy**: forward pass, loss, backward pass, aggiornamento dei pesi
5. **Lo stesso modello con PyTorch**: cosa ci risparmia
6. Una **rete neurale** più grande e il problema dell'**overfitting**
7. **Early stopping**
9. Valutazione sul test set e **classi sbilanciate**
10. Salvare e caricare il modello
11. **Inferenza**: usare il modello su un cliente nuovo
12. **Esercizio**

## 1. Importare le librerie

| Libreria | A cosa serve qui |
|---|---|
| `numpy` | Calcolo numerico con array: lo usiamo per costruire il primo modello a mano |
| `pandas` | Leggere e manipolare tabelle (il file CSV) |
| `matplotlib` | Grafici |
| `torch` | PyTorch: tensori, derivate automatiche, reti neurali |
| `sklearn.metrics` | Metriche finali (matrice di confusione, precision, recall) |
| `copy` | Copiare i pesi del modello migliore (early stopping) |

In [ ]:
import numpy as np                  # array e calcolo numerico
import pandas as pd                 # tabelle (DataFrame): leggere e pulire il CSV
import matplotlib.pyplot as plt     # grafici

import torch                        # tensori + autograd (derivate automatiche)
import torch.nn as nn               # layer e funzioni di loss pronte

# scikit-learn: solo per le metriche di valutazione
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score)

import copy                         # per salvare una copia dei pesi migliori

# Seed fisso: così tutti otteniamo gli stessi risultati
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# Stile dei grafici
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

print("NumPy", np.__version__, "| pandas", pd.__version__, "| PyTorch", torch.__version__)

## 2. Caricare ed esplorare i dati


In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

print("Righe, colonne:", df.shape)
df.head()

In [ ]:
# Tipo di ogni colonna e valori mancanti
df.info()

### Quanti clienti abbandonano?


In [ ]:
conteggi = df["Churn"].value_counts()
print(conteggi)
print("\nPercentuali:\n", (conteggi / len(df) * 100).round(1))

conteggi.plot(kind="bar", color=["tab:blue", "tab:red"])
plt.title("Clienti che abbandonano (Yes) vs che restano (No)")
plt.ylabel("Numero di clienti")
plt.xticks(rotation=0)
plt.show()

### Che cosa influenza l'abbandono?

Prima di costruire un modello, guardiamo i dati: spesso un paio di grafici dicono già molto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Tasso di abbandono per tipo di contratto
tasso = df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean() * 100)
tasso.plot(kind="bar", ax=axes[0], color="tab:orange")
axes[0].set_title("% di abbandono per tipo di contratto")
axes[0].set_ylabel("% churn")
axes[0].tick_params(axis="x", rotation=0)

# Da quanti mesi sono clienti (tenure), divisi per churn
axes[1].hist(df.loc[df["Churn"] == "No", "tenure"], bins=30, alpha=0.6, label="Resta")
axes[1].hist(df.loc[df["Churn"] == "Yes", "tenure"], bins=30, alpha=0.6, label="Abbandona", color="tab:red")
axes[1].set_title("Mesi da cliente (tenure)")
axes[1].set_xlabel("mesi"); axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Preparare i dati

Un modello lavora solo con **numeri**. Dobbiamo:
1. **Pulire**: correggere `TotalCharges` e togliere le colonne inutili
2. **Codificare**: trasformare le colonne di testo in numeri
3. **Dividere** in train / validation / test
4. **Normalizzare** le feature

### 3.1 Pulizia

In [ ]:
# Perché TotalCharges è testo? Cerchiamo i valori che non sono numeri
non_numerici = df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()]
print("Righe con TotalCharges non numerico:", len(non_numerici))
non_numerici[["customerID", "tenure", "TotalCharges"]].head()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # " " -> NaN (valore mancante)
df = df.dropna().reset_index(drop=True)      # sono solo 11 righe su 7043: le eliminiamo
df = df.drop(columns=["customerID"])         # un ID non aiuta a prevedere nulla

print("Nuova shape:", df.shape)

### 3.2 Codificare le colonne di testo (one-hot encoding)

Il modello non capisce `"Month-to-month"`. Usiamo il **one-hot encoding**: ogni categoria diventa una colonna con 0 o 1.

In [ ]:
# Esempio su una sola colonna
print(df["Contract"].head(4).tolist())
pd.get_dummies(df["Contract"].head(4), dtype=int)

In [ ]:
# Target: Yes -> 1, No -> 0
y = (df["Churn"] == "Yes").astype(np.float32).values

# Feature: tutto tranne il target, con le colonne di testo codificate
X_df = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True).astype(np.float32)
feature_cols = list(X_df.columns)       # ci servirà per l'inferenza!
X = X_df.values                         # da DataFrame pandas ad array NumPy

print("X:", X.shape, "| y:", y.shape)
print("Numero di feature:", len(feature_cols))
print(feature_cols)

### 3.3 Dividere in train, validation e test

| Insieme | % | A cosa serve |
|---|---|---|
| **Train** | 70% | Il modello **impara** da questi dati |
| **Validation** | 15% | **Noi** lo usiamo per prendere decisioni (learning rate, architettura, quando fermarsi) |
| **Test** | 15% | Usato **una sola volta alla fine**: stima onesta di come andrà su clienti nuovi |

Analogia, il train è il libro di esercizi, la validation sono le simulazioni d'esame, il test è l'esame vero. Se studi sulle domande dell'esame, il voto non dice nulla su quanto hai imparato.

In [ ]:
n = len(X)
indici = rng.permutation(n)          # mescoliamo gli indici in modo casuale

n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_idx = indici[:n_train]
val_idx = indici[n_train:n_train + n_val]
test_idx = indici[n_train + n_val:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")
print(f"% churn -> train {y_train.mean():.1%} | val {y_val.mean():.1%} | test {y_test.mean():.1%}")

### 3.4 Normalizzare

Le feature hanno scale molto diverse: `TotalCharges` arriva a ~8000, le colonne one-hot sono 0 o 1. Con scale così diverse la discesa del gradiente fa fatica. **Standardizziamo**: ogni feature avrà media 0 e deviazione standard 1.

$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$

> ⚠️ **Media e deviazione standard si calcolano SOLO sul train.** Se usassimo anche validation e test, il modello "sbircerebbe" dati che non dovrebbe vedere (**data leakage**).

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8      # +1e-8 per non dividere mai per zero

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std        # stessi mean e std del train!

print("Prima colonna 'tenure' -> media train:", X_train[:, 1].mean().round(3),
      "| std train:", X_train[:, 1].std().round(3))

## 4. Modello da zero con NumPy: regressione logistica

Partiamo dal modello più semplice per la classificazione binaria: la **regressione logistica**. È un singolo "neurone":

1. **Combinazione lineare:** un peso per ogni feature, più un bias → $z = X w + b$
2. **Sigmoide:** trasforma $z$ in una probabilità tra 0 e 1 → $p = \sigma(z) = \dfrac{1}{1 + e^{-z}}$
Se $p \geq 0.5$ prevediamo *abbandona*, altrimenti *resta*.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -30, 30)            # evita overflow di exp() con numeri molto grandi
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("Funzione sigmoide: da qualsiasi numero a una probabilità")
plt.xlabel("z"); plt.ylabel("σ(z)")
plt.show()

### 4.1 Forward pass

Il **forward pass** è il calcolo "in avanti": dati gli input e i pesi attuali, calcola le predizioni.

In [ ]:
def forward(X, w, b):
    z = X @ w + b          # (n_campioni, n_feature) @ (n_feature,) -> (n_campioni,)
    return sigmoid(z)      # probabilità di churn per ogni cliente

# Proviamo con pesi tutti a zero
n_features = X_train.shape[1]
w = np.zeros(n_features)
b = 0.0

p = forward(X_train[:5], w, b)
print("Probabilità predette:", p)     # tutte 0.5: con pesi nulli il modello non sa nulla

### 4.2 La loss: Binary Cross-Entropy

La **loss** misura **quanto sbaglia** il modello. Per la classificazione binaria si usa la **Binary Cross-Entropy (BCE)**:

$$L = -\frac{1}{n}\sum_{i=1}^{n}\Big[\,y_i \log(p_i) + (1-y_i)\log(1-p_i)\,\Big]$$

- Se il cliente abbandona ($y=1$) e prevediamo $p=0.9$ → loss piccola
- Se il cliente abbandona ($y=1$) e prevediamo $p=0.1$ → loss grande
Più il modello è **sicuro e sbaglia**, più viene punito.

In [ ]:
def bce_loss(p, y):
    p = np.clip(p, 1e-7, 1 - 1e-7)    # evita log(0) = -infinito
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print("Loss con pesi a zero:", bce_loss(forward(X_train, w, b), y_train))
# ≈ 0.693 = log(2): è la loss di chi tira a indovinare con p = 0.5

### 4.3 Backward pass: il gradiente

Per migliorare dobbiamo sapere **in che direzione spostare ogni peso** per far scendere la loss. È il **gradiente**: la derivata della loss rispetto a ogni peso. Il calcolo all'indietro, dalla loss verso i pesi, si chiama **backward pass**.

Per sigmoide + BCE il risultato è sorprendentemente semplice:

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^T (p - y) \qquad\qquad \frac{\partial L}{\partial b} = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)$$

**Intuizione:** $(p - y)$ è l'**errore** su ogni cliente. Ogni peso viene corretto in proporzione all'errore e al valore della sua feature.

<details>
<summary>🔍 <b>Approfondimento: da dove viene questa formula?</b> (clicca per aprire)</summary>
Per un singolo campione: $z = x \cdot w + b$, $\;p = \sigma(z)$, $\;L = -[y\log p + (1-y)\log(1-p)]$.

Usiamo la **regola della catena**: $\dfrac{\partial L}{\partial w} = \dfrac{\partial L}{\partial p}\cdot\dfrac{\partial p}{\partial z}\cdot\dfrac{\partial z}{\partial w}$

**1. Derivata della loss rispetto a p:**
$$\frac{\partial L}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p} = \frac{p - y}{p(1-p)}$$

**2. Derivata della sigmoide** (una proprietà molto comoda):
$$\frac{\partial p}{\partial z} = \sigma(z)\big(1-\sigma(z)\big) = p(1-p)$$

**3. Derivata di z rispetto a w:**
$$\frac{\partial z}{\partial w} = x$$

**Moltiplicando:** il termine $p(1-p)$ si semplifica!
$$\frac{\partial L}{\partial w} = \frac{p - y}{p(1-p)}\cdot p(1-p)\cdot x = (p - y)\,x$$

Facendo la media su tutti gli $n$ campioni e scrivendolo in forma matriciale si ottiene $\frac{1}{n}X^T(p-y)$. Per il bias $\frac{\partial z}{\partial b} = 1$, quindi resta la media di $(p - y)$.

Questa semplificazione non è un caso: sigmoide e cross-entropy sono "fatte l'una per l'altra". È anche il motivo per cui PyTorch offre `BCEWithLogitsLoss`, che le calcola insieme in modo più stabile.
</details>

In [ ]:
def gradients(X, y, p):
    n = len(y)
    errore = p - y                 # (n_campioni,)
    dw = X.T @ errore / n          # (n_feature, n_campioni) @ (n_campioni,) -> (n_feature,)
    db = errore.mean()
    return dw, db

### 4.4 Il ciclo di addestramento (full-batch gradient descent)

Ripetiamo per un certo numero di **epoche**:
1. **Forward**: calcola le predizioni
2. **Loss**: quanto sbagliamo
3. **Backward**: calcola i gradienti
4. **Aggiornamento**: $w \leftarrow w - \text{lr} \cdot \frac{\partial L}{\partial w}$
**Full-batch** significa che a ogni passo usiamo **tutti** i clienti del train per calcolare il gradiente. Un'**epoca** = una passata su tutto il dataset; con il full-batch, un'epoca = un aggiornamento dei pesi.

Il **learning rate** (`lr`) decide **quanto è grande ogni passo**.

In [ ]:
LR = 0.1
EPOCHS = 300

w = np.zeros(n_features)
b = 0.0
storia_np = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    # 1. Forward
    p_train = forward(X_train, w, b)
    # 2. Loss
    loss = bce_loss(p_train, y_train)
    # 3. Backward
    dw, db = gradients(X_train, y_train, p_train)
    # 4. Aggiornamento dei pesi
    w = w - LR * dw
    b = b - LR * db

    # Monitoriamo anche la validation (senza addestrare su di essa!)
    val_loss = bce_loss(forward(X_val, w, b), y_val)
    storia_np["train_loss"].append(loss)
    storia_np["val_loss"].append(val_loss)

    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        print(f"Epoca {epoch:3d} | train loss {loss:.4f} | val loss {val_loss:.4f}")

In [ ]:

plt.plot(storia_np["train_loss"], label="train")
plt.plot(storia_np["val_loss"], label="validation")
plt.title("Regressione logistica (NumPy): andamento della loss")
plt.xlabel("Epoca"); plt.ylabel("BCE loss"); plt.legend()
plt.show()

def accuracy(p, y, soglia=0.5):
    return ((p >= soglia) == y).mean()

print(f"Accuratezza train:      {accuracy(forward(X_train, w, b), y_train):.1%}")
print(f"Accuratezza validation: {accuracy(forward(X_val, w, b), y_val):.1%}")

### 4.5 Che cosa ha imparato il modello?

Il bello della regressione logistica: ogni feature ha **un peso**, quindi possiamo leggerlo.
Peso **positivo** → aumenta la probabilità di abbandono. Peso **negativo** → la riduce.

In [ ]:
pesi = pd.Series(w, index=feature_cols)
top = pesi.reindex(pesi.abs().sort_values(ascending=False).index)[:12][::-1]

colori = ["tab:red" if v > 0 else "tab:blue" for v in top]
top.plot(kind="barh", color=colori, figsize=(8, 5))
plt.title("Le 12 feature più influenti (rosso = aumenta il churn)")
plt.xlabel("Peso")
plt.show()

>Ritroviamo quello che avevamo visto nei grafici: `tenure` alta e contratti lunghi riducono il churn.
> **Attenzione:** con feature correlate (es. `tenure` e `TotalCharges`) i singoli pesi vanno letti con prudenza.

### 4.6 L'importanza del learning rate

Proviamo diversi learning rate. Per comodità mettiamo il ciclo di prima dentro una funzione.

In [ ]:
def train_numpy(lr, epochs=300):
    w = np.zeros(n_features)
    b = 0.0
    losses = []
    for _ in range(epochs):
        p = forward(X_train, w, b)
        losses.append(bce_loss(p, y_train))
        dw, db = gradients(X_train, y_train, p)
        w -= lr * dw
        b -= lr * db
    return losses

for lr in [0.001, 0.01, 0.1, 1.0, 10.0]:
    plt.plot(train_numpy(lr), label=f"lr = {lr}")

plt.title("Effetto del learning rate sulla loss di training")
plt.xlabel("Epoca"); plt.ylabel("BCE loss")
plt.ylim(0.35, 1.2); plt.legend()
plt.show()

| Learning rate | Cosa succede |
|---|---|
| Troppo piccolo (`0.001`) | Impara, ma **lentissimo** |
| Giusto (`0.1` – `1.0`) | Scende velocemente e si stabilizza |
| Troppo grande (`10`) | I passi sono enormi: **salta** oltre il minimo e la loss oscilla |

## 5. Lo stesso modello con PyTorch

Abbiamo scritto a mano: forward, loss, **gradienti** e aggiornamento. Con PyTorch:
- il modello si costruisce con `nn.Linear`
- la loss è già pronta: `nn.BCEWithLogitsLoss`
- **i gradienti li calcola autograd** con `loss.backward()`
- l'aggiornamento lo fa l'**ottimizzatore** con `optimizer.step()`

### 5.1 Da array NumPy a tensori PyTorch


In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

# unsqueeze(1): da (n,) a (n, 1), la stessa shape dell'output del modello
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print(X_train_t.shape, y_train_t.shape)

### 5.2 Il modello come `nn.Module`

In PyTorch ogni modello è una classe che eredita da `nn.Module` e definisce:
- `__init__`: i **layer** (che contengono i pesi)
- `forward`: **come** i dati attraversano i layer

In [ ]:
class RegressioneLogistica(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)   # contiene w (1 x n_feature) e b

    def forward(self, x):
        return self.linear(x)                    # restituisce i "logits" z (senza sigmoide!)

modello_lr = RegressioneLogistica(n_features)

# Pesi a zero, come nella versione NumPy, per poterle confrontare
nn.init.zeros_(modello_lr.linear.weight)
nn.init.zeros_(modello_lr.linear.bias)

print(modello_lr)
print("Parametri:", sum(p.numel() for p in modello_lr.parameters()))

### 5.3 Autograd fa davvero lo stesso calcolo?

Calcoliamo il gradiente con PyTorch e confrontiamolo con la nostra formula $\frac{1}{n}X^T(p-y)$.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()

loss = loss_fn(modello_lr(X_train_t), y_train_t)
loss.backward()                                         # autograd
grad_torch = modello_lr.linear.weight.grad.numpy().ravel()

p0 = forward(X_train, np.zeros(n_features), 0.0)
grad_numpy, _ = gradients(X_train, y_train, p0)         # la nostra formula

print("Differenza massima:", np.abs(grad_torch - grad_numpy).max())
modello_lr.zero_grad()                                  # puliamo prima di addestrare

### 5.4 Il ciclo di addestramento in PyTorch


In [ ]:
optimizer = torch.optim.SGD(modello_lr.parameters(), lr=LR)   # SGD = discesa del gradiente
storia_torch = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    modello_lr.train()
    logits = modello_lr(X_train_t)           # 1. Forward
    loss = loss_fn(logits, y_train_t)        # 2. Loss
    optimizer.zero_grad()                    #    azzera i gradienti del passo precedente
    loss.backward()                          # 3. Backward (autograd)
    optimizer.step()                         # 4. Aggiornamento: w = w - lr * grad

    modello_lr.eval()
    with torch.no_grad():                    # in validazione niente gradienti: più veloce
        val_loss = loss_fn(modello_lr(X_val_t), y_val_t)

    storia_torch["train_loss"].append(loss.item())
    storia_torch["val_loss"].append(val_loss.item())

print(f"Loss finale -> NumPy: {storia_np['train_loss'][-1]:.5f} | PyTorch: {storia_torch['train_loss'][-1]:.5f}")

In [ ]:
plt.plot(storia_np["train_loss"], linewidth=4, alpha=0.4, label="NumPy (a mano)")
plt.plot(storia_torch["train_loss"], "--", color="black", label="PyTorch")
plt.title("Stesso modello, stessi risultati")
plt.xlabel("Epoca"); plt.ylabel("Train loss"); plt.legend()
plt.show()

### 5.5 NumPy vs PyTorch: cosa abbiamo risparmiato

| Passo | NumPy (a mano) | PyTorch |
|---|---|---|
| Modello | `sigmoid(X @ w + b)` | `nn.Linear(n_features, 1)` |
| Loss | funzione `bce_loss` scritta da noi | `nn.BCEWithLogitsLoss()` |
| **Gradienti** | **derivati a mano** (`gradients`) | `loss.backward()` |
| Aggiornamento | `w = w - lr * dw` | `optimizer.step()` |

Per un singolo neurone derivare a mano è fattibile. Per una rete con **più layer** diventerebbe un incubo. **Da qui in poi usiamo solo PyTorch.**

## 6. Una rete neurale più grande e l'overfitting

Una **rete neurale multistrato (MLP)** mette in fila più layer lineari, separati da una **funzione di attivazione** non lineare (qui `ReLU`: $\max(0, z)$). Senza attivazioni, più layer lineari equivalgono a un solo layer lineare.

In [ ]:
def crea_mlp(n_features, hidden=256):
    return nn.Sequential(                 # nn.Sequential: i layer vengono applicati in ordine
        nn.Linear(n_features, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1),
    )

def conta_parametri(modello):
    return sum(p.numel() for p in modello.parameters())

print("Regressione logistica:", conta_parametri(modello_lr), "parametri")
print("MLP:                  ", conta_parametri(crea_mlp(n_features)), "parametri")
print("Clienti nel train:    ", len(X_train))

### 6.1 Una funzione di training riutilizzabile

D'ora in poi addestreremo diversi modelli: mettiamo il ciclo di PyTorch in una funzione. Oltre alla loss, salviamo anche l'**accuratezza** a ogni epoca.

Usiamo **SGD con momentum**: invece di seguire solo il gradiente attuale, l'ottimizzatore "ricorda" la direzione dei passi precedenti, come una palla che rotola giù da una collina e prende velocità. Converge più in fretta.

In [ ]:
def train_model(modello, epochs=1000, lr=0.03, momentum=0.9):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(modello.parameters(), lr=lr, momentum=momentum)
    storia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        # --- training (full-batch) ---
        modello.train()
        logits = modello(X_train_t)
        loss = loss_fn(logits, y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # --- validazione ---
        modello.eval()
        with torch.no_grad():
            val_logits = modello(X_val_t)
            val_loss = loss_fn(val_logits, y_val_t)

        storia["train_loss"].append(loss.item())
        storia["val_loss"].append(val_loss.item())
        storia["train_acc"].append(((logits > 0) == y_train_t).float().mean().item())
        storia["val_acc"].append(((val_logits > 0) == y_val_t).float().mean().item())
        # logits > 0  equivale a  sigmoid(logits) > 0.5

    return modello, storia


def plot_storia(storia, titolo="", best_epoch=None):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(storia["train_loss"], label="train")
    axes[0].plot(storia["val_loss"], label="validation")
    axes[0].set_title(f"Loss {titolo}"); axes[0].set_xlabel("Epoca")
    axes[1].plot(storia["train_acc"], label="train")
    axes[1].plot(storia["val_acc"], label="validation")
    axes[1].set_title(f"Accuratezza {titolo}"); axes[1].set_xlabel("Epoca")
    if best_epoch is not None:
        for ax in axes:
            ax.axvline(best_epoch, color="green", linestyle="--", label=f"migliore (epoca {best_epoch})")
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.show()

### 6.2 Vediamo l'overfitting

Addestriamo l'MLP a lungo, senza fermarci (~30 secondi).


In [ ]:
torch.manual_seed(SEED)
mlp, storia_mlp = train_model(crea_mlp(n_features), epochs=1500, lr=0.03, momentum=0.9)

best = int(np.argmin(storia_mlp["val_loss"]))
plot_storia(storia_mlp, "(MLP, 1500 epoche)", best_epoch=best)

print(f"Val loss minima: {storia_mlp['val_loss'][best]:.4f} all'epoca {best}")
print(f"Val loss finale: {storia_mlp['val_loss'][-1]:.4f}")
print(f"Accuratezza finale -> train {storia_mlp['train_acc'][-1]:.1%} | validation {storia_mlp['val_acc'][-1]:.1%}")

**Come leggere il grafico:**
- La loss di **train** continua a scendere: il modello va sempre meglio sui dati che ha visto.
- La loss di **validation** scende per un po', poi **risale**: sui dati nuovi va sempre **peggio**.
- Da quel momento il modello non sta più imparando regole: sta **memorizzando** i clienti del train.
Questo è l'**overfitting**. Il punto migliore è il minimo della curva di validation (linea verde).

## 7. Early stopping

Idea semplicissima: **fermiamo l'addestramento quando la validation smette di migliorare**, e teniamo i pesi del momento migliore.

- Teniamo traccia della **migliore val loss** vista finora e salviamo una copia dei pesi.
- Se per `patience` epoche consecutive la val loss non migliora, ci fermiamo.
- Alla fine ricarichiamo i pesi migliori.
Aggiorniamo `train_model` aggiungendo l'early stopping. Aggiungiamo già anche due opzioni che useremo più avanti: una **loss personalizzata** (sezione 9) e uno **scheduler** (sezione 8).

In [ ]:
def train_model(modello, epochs=1000, lr=0.03, momentum=0.9, patience=None,
                loss_fn=None, scheduler_step=None, scheduler_gamma=0.5, verbose=True):
    if loss_fn is None:
        loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(modello.parameters(), lr=lr, momentum=momentum)

    scheduler = None
    if scheduler_step is not None:     # sezione 8 (opzionale)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step,
                                                    gamma=scheduler_gamma)

    storia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    miglior_loss = float("inf")
    miglior_pesi = copy.deepcopy(modello.state_dict())
    miglior_epoca = 0
    epoche_senza_miglioramento = 0

    for epoch in range(epochs):
        # --- training (full-batch) ---
        modello.train()
        logits = modello(X_train_t)
        loss = loss_fn(logits, y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        storia["lr"].append(optimizer.param_groups[0]["lr"])
        if scheduler is not None:
            scheduler.step()

        # --- validazione ---
        modello.eval()
        with torch.no_grad():
            val_logits = modello(X_val_t)
            val_loss = loss_fn(val_logits, y_val_t).item()

        storia["train_loss"].append(loss.item())
        storia["val_loss"].append(val_loss)
        storia["train_acc"].append(((logits > 0) == y_train_t).float().mean().item())
        storia["val_acc"].append(((val_logits > 0) == y_val_t).float().mean().item())

        # --- early stopping ---
        if val_loss < miglior_loss:
            miglior_loss = val_loss
            miglior_pesi = copy.deepcopy(modello.state_dict())   # copia dei pesi migliori
            miglior_epoca = epoch
            epoche_senza_miglioramento = 0
        else:
            epoche_senza_miglioramento += 1

        if patience is not None and epoche_senza_miglioramento >= patience:
            if verbose:
                print(f"Early stopping all'epoca {epoch}: nessun miglioramento da {patience} epoche")
            break

    modello.load_state_dict(miglior_pesi)     # torniamo ai pesi migliori
    storia["best_epoch"] = miglior_epoca
    if verbose:
        print(f"Migliore epoca: {miglior_epoca} | val loss: {miglior_loss:.4f}")
    return modello, storia

In [ ]:
torch.manual_seed(SEED)
mlp_es, storia_es = train_model(crea_mlp(n_features), epochs=1500, lr=0.03, momentum=0.9, patience=50)

plot_storia(storia_es, "(MLP + early stopping)", best_epoch=storia_es["best_epoch"])

> Stesso modello, ma l'addestramento si ferma da solo **molto prima delle 1500 epoche**: meno tempo, e teniamo i pesi del punto migliore invece di quelli "sovra-addestrati".

## 8. Learning rate scheduler

Finora il learning rate è rimasto **costante**. Uno **scheduler** lo cambia durante l'addestramento.

**Idea:** all'inizio passi lunghi per avvicinarsi velocemente al minimo, poi passi più corti per "rifinire" senza saltarci sopra.

Usiamo `StepLR`: ogni `step_size` epoche il learning rate viene moltiplicato per `gamma`. Esempio: `lr=0.03`, `step_size=200`, `gamma=0.5` → 0.03 → 0.015 → 0.0075 → …

Per vedere bene l'effetto, **togliamo l'early stopping** e confrontiamo con l'MLP della sezione 6 (stesse 1500 epoche, learning rate costante).

In [ ]:
torch.manual_seed(SEED)
mlp_sched, storia_sched = train_model(crea_mlp(n_features), epochs=1500, lr=0.03, momentum=0.9,
                                      scheduler_step=200, scheduler_gamma=0.5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(storia_sched["lr"])
axes[0].set_title("Learning rate durante il training (StepLR)"); axes[0].set_xlabel("Epoca")
axes[1].plot(storia_mlp["val_loss"], label="lr costante (sezione 6)")
axes[1].plot(storia_sched["val_loss"], label="con StepLR")
axes[1].set_title("Validation loss"); axes[1].set_xlabel("Epoca"); axes[1].legend()
plt.tight_layout()
plt.show()

> Con lo scheduler i passi diventano sempre più piccoli: dopo qualche centinaio di epoche il modello cambia pochissimo, quindi la validation loss **non esplode** come prima. Attenzione però: il **minimo** raggiunto è simile a quello con l'early stopping. Lo scheduler non è magia: è un altro **iperparametro** da provare sul validation set, spesso insieme all'early stopping.

## 9. Valutazione sul test set e classi sbilanciate

Abbiamo preso tutte le decisioni usando il validation set. Ora, **una volta sola**, misuriamo il modello sul **test set**.

Prima di tutto, un confronto onesto: quanto otterrebbe un "modello" che risponde **sempre "No, non abbandona"**?

In [ ]:
def valuta(modello, X_t, y, soglia=0.5):
    """Restituisce le metriche principali per un modello PyTorch."""
    modello.eval()
    with torch.no_grad():
        prob = torch.sigmoid(modello(X_t)).numpy().ravel()   # logits -> probabilità
    pred = (prob >= soglia).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
    }, pred, prob

baseline = 1 - y_test.mean()
metriche_es, pred_es, prob_es = valuta(mlp_es, X_test_t, y_test)

print(f"Baseline 'sempre No': accuracy {baseline:.1%}")
print(f"MLP + early stopping: accuracy {metriche_es['accuracy']:.1%}")

### 9.1 La matrice di confusione

|  | Previsto: resta | Previsto: abbandona |
|---|---|---|
| **Reale: resta** | ✅ Vero negativo (TN) | ❌ Falso positivo (FP): offerta inutile |
| **Reale: abbandona** | ❌ **Falso negativo (FN): cliente perso!** | ✅ Vero positivo (TP) |

- **Precision** = $\frac{TP}{TP + FP}$ → *di quelli che segnaliamo, quanti abbandonano davvero?*
- **Recall** = $\frac{TP}{TP + FN}$ → *di quelli che abbandonano davvero, quanti ne troviamo?*
- **F1** = media armonica di precision e recall

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_es),
                       display_labels=["Resta", "Abbandona"]).plot(cmap="Blues")
plt.title("MLP + early stopping (test set)")
plt.grid(False)
plt.show()

print({k: round(v, 3) for k, v in metriche_es.items()})

> Per l'azienda un **falso negativo** (cliente che se ne va e che non abbiamo visto) costa molto più di un falso positivo (uno sconto dato a chi sarebbe rimasto comunque). Il modello ne trova solo una parte: la **recall** è bassa. Il motivo: vedendo molti più "No" che "Sì", il modello impara a essere prudente.

### 9.2 Soluzione 1: pesare di più la classe rara (`pos_weight`)

Diciamo alla loss che **sbagliare un cliente che abbandona costa di più**. `BCEWithLogitsLoss` ha il parametro `pos_weight`: un valore tipico è il rapporto tra negativi e positivi nel train.

> **È un compromesso:** troviamo molti più clienti a rischio (recall ↑), ma diamo anche più falsi allarmi (precision ↓) e l'accuratezza scende. Quale sia "meglio" non lo decide la matematica: **lo decide il business** (quanto costa un cliente perso rispetto a uno sconto).

### 9.3 Soluzione 2: spostare la soglia di decisione

Non è obbligatorio usare 0.5 come soglia. Abbassandola segnaliamo più clienti; alzandola siamo più selettivi.

> ⚠️ La soglia è una scelta come le altre: la decidiamo sul **validation set**, non sul test.

In [ ]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos])
print(f"pos_weight = {pos_weight.item():.2f}  -> un errore su un 'Sì' pesa ~{pos_weight.item():.1f} volte di più")

torch.manual_seed(SEED)
mlp_pesato, storia_pesato = train_model(crea_mlp(n_features), epochs=1500, lr=0.03, momentum=0.9,
                                        patience=50, loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight))

metriche_pesato, pred_pesato, _ = valuta(mlp_pesato, X_test_t, y_test)

In [ ]:
_, _, prob_val = valuta(mlp_es, X_val_t, y_val)
soglie = np.arange(0.05, 0.96, 0.01)

prec = [precision_score(y_val, prob_val >= s, zero_division=0) for s in soglie]
rec = [recall_score(y_val, prob_val >= s) for s in soglie]
f1s = [f1_score(y_val, prob_val >= s) for s in soglie]

miglior_soglia = float(soglie[np.argmax(f1s)])

plt.plot(soglie, prec, label="Precision")
plt.plot(soglie, rec, label="Recall")
plt.plot(soglie, f1s, label="F1", linewidth=3)
plt.axvline(0.5, color="gray", linestyle=":", label="soglia 0.5")
plt.axvline(miglior_soglia, color="green", linestyle="--", label=f"miglior F1 ({miglior_soglia:.2f})")
plt.title("Precision, recall e F1 al variare della soglia (validation)")
plt.xlabel("Soglia"); plt.legend()
plt.show()

In [ ]:
metriche_soglia, _, _ = valuta(mlp_es, X_test_t, y_test, soglia=miglior_soglia)

pd.DataFrame({
    "Soglia 0.5": metriche_es,
    f"Soglia {miglior_soglia:.2f}": metriche_soglia,
    "pos_weight": metriche_pesato,
}).round(3)

> `pos_weight` e lo spostamento della soglia ottengono un effetto simile per strade diverse: il primo cambia **come impara** il modello, il secondo cambia **come usiamo** le sue probabilità.

---

## 10. Salvare e caricare il modello

Un modello addestrato è utile solo se possiamo **riusarlo** senza riaddestrarlo. In PyTorch si salva lo **`state_dict`**: un dizionario con tutti i pesi.

⚠️ **I pesi da soli non bastano!** Per usare il modello su un cliente nuovo servono anche:
- l'**elenco e l'ordine delle colonne** dopo il one-hot encoding (`feature_cols`)
- **media e deviazione standard** del train (per normalizzare allo stesso modo)
- l'**architettura** (per ricostruire il modello)
- la **soglia** scelta
Salviamo tutto insieme in un unico file (**checkpoint**).

In [ ]:
checkpoint = {
    "model_state": mlp_es.state_dict(),
    "hidden": 256,
    "feature_cols": feature_cols,
    "mean": torch.tensor(mean),
    "std": torch.tensor(std),
    "soglia": miglior_soglia,
}
torch.save(checkpoint, "churn_model.pt")
print("Modello salvato in churn_model.pt")

> Su Colab il file si vede nel pannello **File** (icona della cartella a sinistra). ⚠️ Viene **cancellato quando la sessione si chiude**: per tenerlo si può collegare Google Drive.

Ora facciamo finta di essere in un altro programma: **ricostruiamo** il modello da zero e carichiamo i pesi.

In [ ]:
ckpt = torch.load("churn_model.pt")

modello_caricato = crea_mlp(len(ckpt["feature_cols"]), hidden=ckpt["hidden"])   # stessa architettura
modello_caricato.load_state_dict(ckpt["model_state"])                         # carichiamo i pesi
modello_caricato.eval()                                                       # modalità inferenza

# Verifica: le predizioni sono identiche a quelle del modello originale?
with torch.no_grad():
    uguali = torch.allclose(modello_caricato(X_test_t), mlp_es(X_test_t))
print("Predizioni identiche:", uguali)

## 11. Inferenza: usare il modello su un cliente nuovo

Nella realtà un nuovo cliente arriva come **dati grezzi**, non come vettore normalizzato. Dobbiamo applicare **esattamente la stessa preparazione** del training:

1. Dizionario → DataFrame
2. One-hot encoding
3. **Riallineare le colonne** a `feature_cols` (stesso ordine, colonne mancanti = 0)
4. Normalizzare con media e std **del train**
5. Modello → sigmoide → probabilità

In [ ]:
def prevedi_churn(cliente, ckpt=ckpt, modello=modello_caricato):
    df_nuovo = pd.DataFrame([cliente])

    # get_dummies SENZA drop_first: con un solo cliente drop_first eliminerebbe ogni categoria!
    df_nuovo = pd.get_dummies(df_nuovo)

    # Stesse colonne e stesso ordine del training; quelle che mancano valgono 0
    df_nuovo = df_nuovo.reindex(columns=ckpt["feature_cols"], fill_value=0).astype(np.float32)

    x = torch.tensor(df_nuovo.values)
    x = (x - ckpt["mean"]) / ckpt["std"]

    with torch.no_grad():
        prob = torch.sigmoid(modello(x)).item()
    return prob, prob >= ckpt["soglia"]

In [ ]:
cliente_a_rischio = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes", "StreamingMovies": "Yes",
    "Contract": "Month-to-month", "PaperlessBilling": "Yes", "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.0, "TotalCharges": 190.0,
}

cliente_fedele = {
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "Yes",
    "tenure": 60, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
    "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No", "StreamingMovies": "No",
    "Contract": "Two year", "PaperlessBilling": "No", "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 65.0, "TotalCharges": 3900.0,
}

for nome, cliente in [("Cliente a rischio", cliente_a_rischio), ("Cliente fedele", cliente_fedele)]:
    prob, abbandona = prevedi_churn(cliente)
    print(f"{nome:18s} -> probabilità di churn {prob:.1%} | {'⚠️ A RISCHIO' if abbandona else '✅ ok'}")

## 12. Esercizio

Partendo dal template qui sotto, costruite un modello diverso e cercate di **battere l'MLP con early stopping sul validation set**.

**Idee da provare (una alla volta, per capire l'effetto di ciascuna):**

| Idea | Come |
|---|---|
| Più o meno neuroni | cambiate `64` in `16`, `128`, `512` |
| Più layer | aggiungete altre coppie `nn.Linear(...)`, `nn.ReLU()` |
| **Dropout** | aggiungete `nn.Dropout(0.3)` dopo una `ReLU`: durante il training "spegne" a caso il 30% dei neuroni, così la rete non può affidarsi troppo a pochi neuroni → meno overfitting |
| Altra attivazione | `nn.Tanh()`, `nn.LeakyReLU()` al posto di `nn.ReLU()` |
| Learning rate e momentum | `lr=0.01`, `lr=0.1`, `momentum=0.0` |
| Classi sbilanciate | `loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight)` |
| Scheduler | `scheduler_step=100, scheduler_gamma=0.5` |

> ⚠️ **Regola:** confrontate i modelli **solo sul validation set**. Il test set si usa alla fine, una volta, sul modello scelto.

In [ ]:
# ===================== ESERCIZIO =====================
torch.manual_seed(SEED)

mio_modello = nn.Sequential(
    nn.Linear(n_features, 64),
    nn.ReLU(),
    # TODO: aggiungete layer, Dropout, cambiate attivazione...
    nn.Linear(64, 1),
)

mio_modello, mia_storia = train_model(
    mio_modello,
    epochs=1500,
    lr=0.03,         # TODO: provate altri valori
    momentum=0.9,
    patience=50,
)

plot_storia(mia_storia, "(il mio modello)", best_epoch=mia_storia["best_epoch"])

In [ ]:
# Confronto sul VALIDATION set
risultati = {
    "Regressione logistica": valuta(modello_lr, X_val_t, y_val)[0],
    "MLP + early stopping": valuta(mlp_es, X_val_t, y_val)[0],
    "MLP + pos_weight": valuta(mlp_pesato, X_val_t, y_val)[0],
    "Il mio modello": valuta(mio_modello, X_val_t, y_val)[0],
}

tabella = pd.DataFrame(risultati).T.round(3)
tabella.plot(kind="bar", figsize=(10, 4), rot=0, ylim=(0, 1))
plt.title("Confronto dei modelli (validation set)")
plt.legend(loc="lower right")
plt.show()

tabella